# Day 5 — Building a Company Brochure with LLMs

## Business Challenge

Create a product that takes a **company name** and its **primary website**, then builds a short brochure for prospective **customers, investors, and recruits**.

### Day 5 workflow

1. Fetch links from the company's website.
2. Use an LLM to decide which links are relevant.
3. Fetch the landing page and relevant pages.
4. Assemble the information into a prompt.
5. Use another LLM call to generate the brochure.
6. Display the brochure as Markdown.
7. Stream the brochure for a typewriter-style experience.

> This extends the Day 1 work into a multi-call LLM application.


## Key concepts from the lesson

### 1. One-shot prompting

The link-selection prompt gives the model an example of the JSON structure expected in the response.

### 2. Structured JSON

The first LLM call is asked to return relevant links in JSON:

```json
{
    "links": [
        {"type": "about page", "url": "https://full.url/about"},
        {"type": "careers page", "url": "https://full.url/careers"}
    ]
}
```

### 3. Multiple LLM calls

The application uses one LLM call to select useful pages and another LLM call to generate the final brochure.

This is an early example of an **Agentic AI design pattern**: combining multiple LLM calls to accomplish a larger task.

### 4. Streaming

The final brochure can be streamed back incrementally to create a familiar typewriter-style effect.


## Imports and environment

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

from google import genai
from google.genai import types

from web_scraper import fetch_website_links, fetch_website_contents


In [ ]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini client created")


## Step 1 — Fetch website links

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links


## Step 2 — Tell the LLM which links are relevant

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))


## Step 3 — Select relevant links with Gemini

The Gemini version keeps the same lesson structure:

- `system_instruction` → system-level instruction
- `contents` → user prompt/content
- `response_mime_type="application/json"` → ask for JSON output
- `response.text` → generated text
- `json.loads()` → convert the JSON string into a Python object


In [ ]:
def select_relevant_links(url):
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=get_links_user_prompt(url),
        config=types.GenerateContentConfig(
            system_instruction=link_system_prompt,
            response_mime_type="application/json"
        )
    )

    result = response.text
    links = json.loads(result)

    return links


In [ ]:
select_relevant_links("https://huggingface.co")


## OpenAI version from the instructor notebook

The original lesson uses the OpenAI Chat Completions interface for the same step:

```python
response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(url)}
    ],
    response_format={"type": "json_object"}
)

result = response.choices[0].message.content
links = json.loads(result)
```

The important lesson is the **same workflow**, even though the provider's SDK syntax is different.


## Step 4 — Fetch the landing page and all relevant links

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])

    return result


In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))


## Step 5 — Create the brochure prompt

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""

    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]

    return user_prompt


In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")


## Step 6 — Generate the brochure with Gemini

Here the practice version requests JSON and then extracts the Markdown field.

The important distinction I learned while practicing:

- If the model returns JSON containing Markdown, parse the JSON first.
- `Markdown()` only controls how Jupyter displays a string; it does not tell the model to generate Markdown.


In [ ]:
def create_brochure(company_name, url):
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=get_brochure_user_prompt(company_name, url),
        config=types.GenerateContentConfig(
            system_instruction=brochure_system_prompt,
            response_mime_type="application/json"
        )
    )

    result = json.loads(response.text)

    display(Markdown(result["brochure_markdown"]))


In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")


## Streaming the brochure with Gemini

### Core Gemini streaming pattern

```python
stream = client.models.generate_content_stream(
    model="gemini-3.6-flash",
    contents="Explain LLMs"
)

for chunk in stream:
    print(chunk.text, end="")
```

The key memory pattern is:

**`generate_content_stream()` → `chunk.text`**

The `display()`, `Markdown()`, and `update_display()` calls below are Jupyter display logic used to create the typewriter-style effect.


In [ ]:
def stream_brochure(company_name, url):
    stream = client.models.generate_content_stream(
        model="gemini-3.6-flash",
        contents=get_brochure_user_prompt(company_name, url),
        config=types.GenerateContentConfig(
            system_instruction=brochure_system_prompt
        )
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.text or ""
        update_display(
            Markdown(response),
            display_id=display_handle.display_id
        )


In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")


## Day 5 Recall Sheet

### Business problem
Build a company brochure from a company name + primary website.

### Flow

```text
Company URL
    ↓
Fetch website links
    ↓
LLM selects relevant links
    ↓
Fetch landing page + relevant pages
    ↓
Combine the information
    ↓
LLM generates brochure
    ↓
Markdown output
```

### Important things to remember

- **One-shot prompting**: provide an example of the expected JSON response.
- **Structured JSON**: use JSON when the application needs structured data from the model.
- **Multiple LLM calls**: one call selects useful links; another creates the brochure.
- **Gemini normal generation**: `client.models.generate_content()`
- **Gemini streaming**: `client.models.generate_content_stream()`
- **Gemini stream chunk**: `chunk.text`
- **Markdown display**: `Markdown(...)` controls Jupyter rendering.
- If JSON contains a Markdown field: `json.loads(response.text)` → `result["brochure_markdown"]`.
- Streaming the final Markdown directly is simpler when JSON structure is not required.

### Main lesson

LLMs can be combined with normal Python/web-scraping code to build a useful business application rather than using an LLM only for a single question.


## My Day 5 Practice

I implemented the instructor's company-brochure workflow using the Gemini Python client and practiced:

- Gemini API client setup with `.env`
- `client.models.generate_content()`
- System instructions + user contents
- JSON responses and `json.loads()`
- Extracting `brochure_markdown`
- `client.models.generate_content_stream()`
- Streaming with `chunk.text`
- Updating Markdown output inside Jupyter

This notebook is intentionally limited to the Day 5 lesson and the related practice work.
